# 🎓 SABCQ AI Counselor - MuseTalk Cloud GPU Engine (v3.0 Phase 1)
### ⚡ Warm MuseTalk 1.5 + chunk-safe lip-sync on Free Google Colab T4

**Phase 1:** models stay loaded in GPU RAM (no cold subprocess every turn).
Long intros are rendered as short chunks by the Nest backend.

---
### 🚀 Quick Start:
1. Upload `priya_sharma_base.mp4`.
2. **Runtime ➔ Run all** (`Ctrl + F9`).
3. Copy Cloudflare URL into `aiqtest_backend/.env` as `AVATAR_SERVICE_URL=...`
4. `/health` should show `warm_mode: warm` after warmup.


## 1. Verify GPU Allocation & Setup Clean Python 3.10 Environment

In [ ]:
# 1. Verify GPU Allocation & Setup Clean Python 3.10 with Micromamba
!nvidia-smi
import os, sys

print("⚡ Installing Micromamba & Python 3.10...")
if not os.path.exists("/usr/local/bin/micromamba"):
    !wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
    !mv bin/micromamba /usr/local/bin/micromamba

if not os.path.exists("/root/.local/share/mamba/envs/py310/bin/python"):
    !micromamba create -y -n py310 python=3.10 -c conda-forge

# Create symlinks so all paths work reliably
!mkdir -p /usr/local/micromamba/envs && ln -sfn /root/.local/share/mamba/envs/py310 /usr/local/micromamba/envs/py310

ENV_DIR = "/root/.local/share/mamba/envs/py310" if os.path.exists("/root/.local/share/mamba/envs/py310") else "/usr/local/micromamba/envs/py310"
ENV_PY = f"{ENV_DIR}/bin/python"
ENV_PIP = f"{ENV_DIR}/bin/pip"
ENV_MIM = f"{ENV_DIR}/bin/mim"

!$ENV_PY --version
print("✅ Clean Python 3.10 environment ready at:", ENV_DIR)

## 2. Clone MuseTalk & Install Compatible PyTorch 2.0.1 + MMPose + Download Weights

In [ ]:
# 2. Clone MuseTalk & Install Dependencies & Model Weights
import os
ENV_DIR = "/root/.local/share/mamba/envs/py310" if os.path.exists("/root/.local/share/mamba/envs/py310") else "/usr/local/micromamba/envs/py310"
ENV_PY = f"{ENV_DIR}/bin/python"
ENV_PIP = f"{ENV_DIR}/bin/pip"
ENV_MIM = f"{ENV_DIR}/bin/mim"

if not os.path.exists("/content/MuseTalk"):
    !git clone https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk

%cd /content/MuseTalk

print("📦 Installing PyTorch with CUDA 11.8 into Python 3.10...")
!$ENV_PIP install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

print("📦 Installing OpenMMLab packages (MMCV, MMPose, MMDetection)...")
!$ENV_PIP install -q "numpy<2"
!$ENV_PIP install -q -U openmim
!$ENV_MIM install mmengine "mmcv==2.0.1" "mmdet==3.1.0"
!$ENV_PIP install -q "mmpose==1.1.0" --no-deps
!$ENV_PIP install -q xtcocotools json_tricks munkres matplotlib-inline

print("📦 Installing MuseTalk dependencies...")
!$ENV_PIP install -q -r requirements.txt
!$ENV_PIP install -q diffusers accelerate opencv-python soundfile librosa fastapi uvicorn python-multipart pycloudflared pyyaml huggingface_hub gdown
!apt-get install -y -qq ffmpeg

print("📥 Downloading MuseTalk Weights...")
weights_script = '''import os
from huggingface_hub import hf_hub_download
for d in ["models/musetalkV15", "models/musetalk", "models/sd-vae", "models/whisper", "models/dwpose", "models/syncnet", "models/face-parse-bisent"]:
    os.makedirs(d, exist_ok=True)
hf_hub_download("TMElyralab/MuseTalk", "musetalkV15/unet.pth", local_dir="models")
hf_hub_download("TMElyralab/MuseTalk", "musetalkV15/musetalk.json", local_dir="models")
hf_hub_download("TMElyralab/MuseTalk", "musetalk/musetalk.json", local_dir="models")
hf_hub_download("TMElyralab/MuseTalk", "musetalk/pytorch_model.bin", local_dir="models")
hf_hub_download("stabilityai/sd-vae-ft-mse", "config.json", local_dir="models/sd-vae")
hf_hub_download("stabilityai/sd-vae-ft-mse", "diffusion_pytorch_model.bin", local_dir="models/sd-vae")
hf_hub_download("openai/whisper-tiny", "config.json", local_dir="models/whisper")
hf_hub_download("openai/whisper-tiny", "pytorch_model.bin", local_dir="models/whisper")
hf_hub_download("openai/whisper-tiny", "preprocessor_config.json", local_dir="models/whisper")
hf_hub_download("yzd-v/DWPose", "dw-ll_ucoco_384.pth", local_dir="models/dwpose")
hf_hub_download("ByteDance/LatentSync", "latentsync_syncnet.pt", local_dir="models/syncnet")
'''
with open("/content/MuseTalk/download_weights.py", "w") as f:
    f.write(weights_script)

!$ENV_PY /content/MuseTalk/download_weights.py

if not os.path.exists("models/face-parse-bisent/79999_iter.pth"):
    !gdown --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O models/face-parse-bisent/79999_iter.pth
if not os.path.exists("models/face-parse-bisent/resnet18-5c106cde.pth"):
    !curl -s -L https://download.pytorch.org/models/resnet18-5c106cde.pth -o models/face-parse-bisent/resnet18-5c106cde.pth

print("✅ All dependencies & model weights ready!")


## 3. Setup Priya Sharma Avatar & Pre-compute Facial Landmarks (coords.pkl)

In [ ]:
# 3. Setup Priya Sharma Avatar & Pre-compute Landmarks
import os, yaml
%cd /content/MuseTalk
ENV_DIR = "/root/.local/share/mamba/envs/py310" if os.path.exists("/root/.local/share/mamba/envs/py310") else "/usr/local/micromamba/envs/py310"
ENV_PY = f"{ENV_DIR}/bin/python"

os.makedirs("/content/MuseTalk/data/video", exist_ok=True)
os.makedirs("/content/MuseTalk/data/audio", exist_ok=True)
AVATAR_PATH = "/content/MuseTalk/data/video/priya_sharma_base.mp4"

# Check if user placed priya_sharma_base.mp4 in /content/
if os.path.exists("/content/priya_sharma_base.mp4"):
    print("✅ Found priya_sharma_base.mp4 in /content/")
    !cp /content/priya_sharma_base.mp4 {AVATAR_PATH}
elif not os.path.exists(AVATAR_PATH):
    print("📢 priya_sharma_base.mp4 not found in /content/")
    print("👉 Click 'Choose Files' below to upload 'priya_sharma_base.mp4':")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".mp4"):
            !cp "{fn}" {AVATAR_PATH}
            break

if not os.path.exists(AVATAR_PATH):
    print("⚠️ Video not uploaded, falling back to default monalisa.mp4")
    !cp /content/MuseTalk/data/video/monalisa.mp4 {AVATAR_PATH}

SAMPLE_AUDIO = "data/audio/sample_clip.wav"
if not os.path.exists(SAMPLE_AUDIO):
    !ffmpeg -y -i {AVATAR_PATH} -t 3 -ar 16000 -ac 1 {SAMPLE_AUDIO} -loglevel error || true

cfg = {
    "avator_1": {
        "preparation": True,
        "bbox_shift": 0,
        "video_path": AVATAR_PATH,
        "audio_clips": {
            "audio_0": SAMPLE_AUDIO
        }
    }
}
os.makedirs("configs/inference", exist_ok=True)
with open("configs/inference/realtime.yaml", "w") as f:
    yaml.dump(cfg, f)

if os.path.exists("results/v15/avatars/avator_1/coords.pkl"):
    print("⚡ Priya Sharma landmarks already cached! Ready for instant inference.")
else:
    print("⚡ Ensuring MMPose helper packages (xtcocotools)...")
    !$ENV_PIP install -q xtcocotools json_tricks munkres
    print("⚡ Extracting Priya Sharma facial landmarks & bounding boxes...")
    !MPLBACKEND=Agg $ENV_PY -m scripts.realtime_inference --inference_config configs/inference/realtime.yaml --version v15

if os.path.exists("results/v15/avatars/avator_1/coords.pkl"):
    print("✅ Priya Sharma cached successfully! Latents are ready for instant generation!")
else:
    print("⚠️ Notice: Landmark file coords.pkl not found yet.")


## 4. Write SABCQ MuseTalk Live Server Code

In [ ]:
# 4. Write SABCQ Warm Engine + Live Server (v3.0 Phase 1)
%cd /content/MuseTalk
import base64
WARM_B64 = "IiIiClNBQkNRIE11c2VUYWxrIDEuNSBXYXJtIEVuZ2luZSAoUGhhc2UgMSkKCkxvYWRzIFZBRSAvIFVOZXQgLyBXaGlzcGVyIC8gQXZhdGFyIGxhdGVudHMgT05DRSBpbnRvIEdQVSBtZW1vcnksIHRoZW4Kc2VydmVzIG1hbnkgYXVkaW/ihpJ2aWRlbyBqb2JzIHdpdGhvdXQgY29sZCBzdWJwcm9jZXNzIHJlbG9hZHMuCgpNdXN0IHJ1biBmcm9tIC9jb250ZW50L011c2VUYWxrIChvciBhbnkgTXVzZVRhbGsgcmVwbyByb290KSBzbyBpbXBvcnRzIHJlc29sdmUuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGNvcHkKaW1wb3J0IGdsb2IKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBwaWNrbGUKaW1wb3J0IHF1ZXVlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGhyZWFkaW5nCmltcG9ydCB0aW1lCmZyb20gdHlwaW5nIGltcG9ydCBPcHRpb25hbAoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdHFkbSBpbXBvcnQgdHFkbQpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgV2hpc3Blck1vZGVsCgoKZGVmIF9mYXN0X2NoZWNrX2ZmbXBlZygpIC0+IGJvb2w6CiAgICB0cnk6CiAgICAgICAgc3VicHJvY2Vzcy5ydW4oWyJmZm1wZWciLCAiLXZlcnNpb24iXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgY2hlY2s9VHJ1ZSkKICAgICAgICByZXR1cm4gVHJ1ZQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gRmFsc2UKCgpjbGFzcyBXYXJtTXVzZVRhbGtFbmdpbmU6CiAgICAiIiJJbi1wcm9jZXNzIE11c2VUYWxrIDEuNSByZWFsdGltZSBhdmF0YXIg4oCUIHdhcm0gbW9kZWxzICsgY2FjaGVkIFByaXlhIGxhdGVudHMuIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgYXZhdGFyX2lkOiBzdHIgPSAiYXZhdG9yXzEiLAogICAgICAgIHZpZGVvX3BhdGg6IHN0ciA9ICJkYXRhL3ZpZGVvL3ByaXlhX3NoYXJtYV9iYXNlLm1wNCIsCiAgICAgICAgYmJveF9zaGlmdDogaW50ID0gMCwKICAgICAgICBiYXRjaF9zaXplOiBpbnQgPSAxNiwKICAgICAgICB2ZXJzaW9uOiBzdHIgPSAidjE1IiwKICAgICAgICBmcHM6IGludCA9IDI1LAogICAgICAgIGdwdV9pZDogaW50ID0gMCwKICAgICAgICBleHRyYV9tYXJnaW46IGludCA9IDEwLAogICAgKToKICAgICAgICBzZWxmLmF2YXRhcl9pZCA9IGF2YXRhcl9pZAogICAgICAgIHNlbGYudmlkZW9fcGF0aCA9IHZpZGVvX3BhdGgKICAgICAgICBzZWxmLmJib3hfc2hpZnQgPSBpbnQoYmJveF9zaGlmdCkKICAgICAgICBzZWxmLmJhdGNoX3NpemUgPSBpbnQoYmF0Y2hfc2l6ZSkKICAgICAgICBzZWxmLnZlcnNpb24gPSB2ZXJzaW9uCiAgICAgICAgc2VsZi5mcHMgPSBpbnQoZnBzKQogICAgICAgIHNlbGYuZ3B1X2lkID0gaW50KGdwdV9pZCkKICAgICAgICBzZWxmLmV4dHJhX21hcmdpbiA9IGludChleHRyYV9tYXJnaW4pCiAgICAgICAgc2VsZi5yZWFkeSA9IEZhbHNlCiAgICAgICAgc2VsZi5fbG9jayA9IHRocmVhZGluZy5Mb2NrKCkKICAgICAgICBzZWxmLl9sb2FkX2Vycm9yOiBPcHRpb25hbFtzdHJdID0gTm9uZQoKICAgICAgICAjIHBvcHVsYXRlZCBieSBsb2FkKCkKICAgICAgICBzZWxmLmRldmljZSA9IE5vbmUKICAgICAgICBzZWxmLnZhZSA9IE5vbmUKICAgICAgICBzZWxmLnVuZXQgPSBOb25lCiAgICAgICAgc2VsZi5wZSA9IE5vbmUKICAgICAgICBzZWxmLnRpbWVzdGVwcyA9IE5vbmUKICAgICAgICBzZWxmLndoaXNwZXIgPSBOb25lCiAgICAgICAgc2VsZi5hdWRpb19wcm9jZXNzb3IgPSBOb25lCiAgICAgICAgc2VsZi53ZWlnaHRfZHR5cGUgPSBOb25lCiAgICAgICAgc2VsZi5mcCA9IE5vbmUKICAgICAgICBzZWxmLmlucHV0X2xhdGVudF9saXN0X2N5Y2xlID0gTm9uZQogICAgICAgIHNlbGYuY29vcmRfbGlzdF9jeWNsZSA9IE5vbmUKICAgICAgICBzZWxmLmZyYW1lX2xpc3RfY3ljbGUgPSBOb25lCiAgICAgICAgc2VsZi5tYXNrX2xpc3RfY3ljbGUgPSBOb25lCiAgICAgICAgc2VsZi5tYXNrX2Nvb3Jkc19saXN0X2N5Y2xlID0gTm9uZQogICAgICAgIHNlbGYuYXZhdGFyX3BhdGggPSBOb25lCiAgICAgICAgc2VsZi52aWRlb19vdXRfcGF0aCA9IE5vbmUKCiAgICBkZWYgbG9hZChzZWxmKSAtPiBib29sOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJvbSBtdXNldGFsay51dGlscy5mYWNlX3BhcnNpbmcgaW1wb3J0IEZhY2VQYXJzaW5nCiAgICAgICAgICAgIGZyb20gbXVzZXRhbGsudXRpbHMudXRpbHMgaW1wb3J0IGxvYWRfYWxsX21vZGVsCiAgICAgICAgICAgIGZyb20gbXVzZXRhbGsudXRpbHMuYXVkaW9fcHJvY2Vzc29yIGltcG9ydCBBdWRpb1Byb2Nlc3NvcgogICAgICAgICAgICBmcm9tIG11c2V0YWxrLnV0aWxzLnByZXByb2Nlc3NpbmcgaW1wb3J0IHJlYWRfaW1ncwogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgc2VsZi5fbG9hZF9lcnJvciA9IGYiTXVzZVRhbGsgaW1wb3J0cyBmYWlsZWQ6IHtlfSIKICAgICAgICAgICAgcHJpbnQoZiJbV2FybUVuZ2luZV0ge3NlbGYuX2xvYWRfZXJyb3J9IikKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgICAgIGlmIG5vdCBfZmFzdF9jaGVja19mZm1wZWcoKToKICAgICAgICAgICAgc2VsZi5fbG9hZF9lcnJvciA9ICJmZm1wZWcgbm90IGZvdW5kIG9uIFBBVEgiCiAgICAgICAgICAgIHByaW50KGYiW1dhcm1FbmdpbmVdIHtzZWxmLl9sb2FkX2Vycm9yfSIpCiAgICAgICAgICAgIHJldHVybiBGYWxzZQoKICAgICAgICAjIFBhdGhzIGZvciB2MS41IHdlaWdodHMKICAgICAgICBpZiBzZWxmLnZlcnNpb24gPT0gInYxNSI6CiAgICAgICAgICAgIHVuZXRfbW9kZWxfcGF0aCA9ICIuL21vZGVscy9tdXNldGFsa1YxNS91bmV0LnB0aCIKICAgICAgICAgICAgdW5ldF9jb25maWcgPSAiLi9tb2RlbHMvbXVzZXRhbGtWMTUvbXVzZXRhbGsuanNvbiIKICAgICAgICAgICAgaWYgbm90IG9zLnBhdGguZXhpc3RzKHVuZXRfbW9kZWxfcGF0aCk6CiAgICAgICAgICAgICAgICAjIGZhbGxiYWNrIG5hbWVzIHNvbWUgY2xvbmVzIHVzZQogICAgICAgICAgICAgICAgdW5ldF9tb2RlbF9wYXRoID0gIi4vbW9kZWxzL211c2V0YWxrL3B5dG9yY2hfbW9kZWwuYmluIgogICAgICAgICAgICAgICAgdW5ldF9jb25maWcgPSAiLi9tb2RlbHMvbXVzZXRhbGsvbXVzZXRhbGsuanNvbiIKICAgICAgICAgICAgc2VsZi5hdmF0YXJfcGF0aCA9IGYiLi9yZXN1bHRzL3YxNS9hdmF0YXJzL3tzZWxmLmF2YXRhcl9pZH0iCiAgICAgICAgZWxzZToKICAgICAgICAgICAgdW5ldF9tb2RlbF9wYXRoID0gIi4vbW9kZWxzL211c2V0YWxrL3B5dG9yY2hfbW9kZWwuYmluIgogICAgICAgICAgICB1bmV0X2NvbmZpZyA9ICIuL21vZGVscy9tdXNldGFsay9tdXNldGFsay5qc29uIgogICAgICAgICAgICBzZWxmLmF2YXRhcl9wYXRoID0gZiIuL3Jlc3VsdHMvYXZhdGFycy97c2VsZi5hdmF0YXJfaWR9IgoKICAgICAgICBjb29yZHNfcGF0aCA9IGYie3NlbGYuYXZhdGFyX3BhdGh9L2Nvb3Jkcy5wa2wiCiAgICAgICAgbGF0ZW50c19wYXRoID0gZiJ7c2VsZi5hdmF0YXJfcGF0aH0vbGF0ZW50cy5wdCIKICAgICAgICBtYXNrX2Nvb3Jkc19wYXRoID0gZiJ7c2VsZi5hdmF0YXJfcGF0aH0vbWFza19jb29yZHMucGtsIgogICAgICAgIGZ1bGxfaW1nc19wYXRoID0gZiJ7c2VsZi5hdmF0YXJfcGF0aH0vZnVsbF9pbWdzIgogICAgICAgIG1hc2tfb3V0X3BhdGggPSBmIntzZWxmLmF2YXRhcl9wYXRofS9tYXNrIgogICAgICAgIHNlbGYudmlkZW9fb3V0X3BhdGggPSBmIntzZWxmLmF2YXRhcl9wYXRofS92aWRfb3V0cHV0LyIKICAgICAgICBvcy5tYWtlZGlycyhzZWxmLnZpZGVvX291dF9wYXRoLCBleGlzdF9vaz1UcnVlKQoKICAgICAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoY29vcmRzX3BhdGgpIG9yIG5vdCBvcy5wYXRoLmV4aXN0cyhsYXRlbnRzX3BhdGgpOgogICAgICAgICAgICBzZWxmLl9sb2FkX2Vycm9yID0gKAogICAgICAgICAgICAgICAgZiJBdmF0YXIgY2FjaGUgbWlzc2luZyB1bmRlciB7c2VsZi5hdmF0YXJfcGF0aH0uICIKICAgICAgICAgICAgICAgICJSdW4gQ29sYWIgQ2VsbCAzIHByZXBhcmF0aW9uIGZpcnN0LiIKICAgICAgICAgICAgKQogICAgICAgICAgICBwcmludChmIltXYXJtRW5naW5lXSB7c2VsZi5fbG9hZF9lcnJvcn0iKQogICAgICAgICAgICByZXR1cm4gRmFsc2UKCiAgICAgICAgcHJpbnQoIltXYXJtRW5naW5lXSBMb2FkaW5nIE11c2VUYWxrIDEuNSBtb2RlbHMgb250byBHUFUgKG9uZS10aW1lKeKApiIpCiAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgIHNlbGYuZGV2aWNlID0gdG9yY2guZGV2aWNlKAogICAgICAgICAgICBmImN1ZGE6e3NlbGYuZ3B1X2lkfSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiCiAgICAgICAgKQoKICAgICAgICBzZWxmLnZhZSwgc2VsZi51bmV0LCBzZWxmLnBlID0gbG9hZF9hbGxfbW9kZWwoCiAgICAgICAgICAgIHVuZXRfbW9kZWxfcGF0aD11bmV0X21vZGVsX3BhdGgsCiAgICAgICAgICAgIHZhZV90eXBlPSJzZC12YWUiLAogICAgICAgICAgICB1bmV0X2NvbmZpZz11bmV0X2NvbmZpZywKICAgICAgICAgICAgZGV2aWNlPXNlbGYuZGV2aWNlLAogICAgICAgICkKICAgICAgICBzZWxmLnRpbWVzdGVwcyA9IHRvcmNoLnRlbnNvcihbMF0sIGRldmljZT1zZWxmLmRldmljZSkKICAgICAgICBzZWxmLnBlID0gc2VsZi5wZS5oYWxmKCkudG8oc2VsZi5kZXZpY2UpCiAgICAgICAgc2VsZi52YWUudmFlID0gc2VsZi52YWUudmFlLmhhbGYoKS50byhzZWxmLmRldmljZSkKICAgICAgICBzZWxmLnVuZXQubW9kZWwgPSBzZWxmLnVuZXQubW9kZWwuaGFsZigpLnRvKHNlbGYuZGV2aWNlKQoKICAgICAgICB3aGlzcGVyX2RpciA9ICIuL21vZGVscy93aGlzcGVyIgogICAgICAgIHNlbGYuYXVkaW9fcHJvY2Vzc29yID0gQXVkaW9Qcm9jZXNzb3IoZmVhdHVyZV9leHRyYWN0b3JfcGF0aD13aGlzcGVyX2RpcikKICAgICAgICBzZWxmLndlaWdodF9kdHlwZSA9IHNlbGYudW5ldC5tb2RlbC5kdHlwZQogICAgICAgIHNlbGYud2hpc3BlciA9IFdoaXNwZXJNb2RlbC5mcm9tX3ByZXRyYWluZWQod2hpc3Blcl9kaXIpCiAgICAgICAgc2VsZi53aGlzcGVyID0gc2VsZi53aGlzcGVyLnRvKGRldmljZT1zZWxmLmRldmljZSwgZHR5cGU9c2VsZi53ZWlnaHRfZHR5cGUpLmV2YWwoKQogICAgICAgIHNlbGYud2hpc3Blci5yZXF1aXJlc19ncmFkXyhGYWxzZSkKCiAgICAgICAgaWYgc2VsZi52ZXJzaW9uID09ICJ2MTUiOgogICAgICAgICAgICBzZWxmLmZwID0gRmFjZVBhcnNpbmcobGVmdF9jaGVla193aWR0aD05MCwgcmlnaHRfY2hlZWtfd2lkdGg9OTApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5mcCA9IEZhY2VQYXJzaW5nKCkKCiAgICAgICAgcHJpbnQoIltXYXJtRW5naW5lXSBMb2FkaW5nIGNhY2hlZCBQcml5YSBTaGFybWEgbGF0ZW50cyAvIGZyYW1lc+KApiIpCiAgICAgICAgc2VsZi5pbnB1dF9sYXRlbnRfbGlzdF9jeWNsZSA9IHRvcmNoLmxvYWQobGF0ZW50c19wYXRoLCBtYXBfbG9jYXRpb249ImNwdSIpCiAgICAgICAgd2l0aCBvcGVuKGNvb3Jkc19wYXRoLCAicmIiKSBhcyBmOgogICAgICAgICAgICBzZWxmLmNvb3JkX2xpc3RfY3ljbGUgPSBwaWNrbGUubG9hZChmKQogICAgICAgIHdpdGggb3BlbihtYXNrX2Nvb3Jkc19wYXRoLCAicmIiKSBhcyBmOgogICAgICAgICAgICBzZWxmLm1hc2tfY29vcmRzX2xpc3RfY3ljbGUgPSBwaWNrbGUubG9hZChmKQoKICAgICAgICBpbnB1dF9pbWdfbGlzdCA9IHNvcnRlZCgKICAgICAgICAgICAgZ2xvYi5nbG9iKG9zLnBhdGguam9pbihmdWxsX2ltZ3NfcGF0aCwgIiouW2pwSlBdW3BuUE5dKltnR10iKSksCiAgICAgICAgICAgIGtleT1sYW1iZGEgeDogaW50KG9zLnBhdGguc3BsaXRleHQob3MucGF0aC5iYXNlbmFtZSh4KSlbMF0pLAogICAgICAgICkKICAgICAgICBzZWxmLmZyYW1lX2xpc3RfY3ljbGUgPSByZWFkX2ltZ3MoaW5wdXRfaW1nX2xpc3QpCiAgICAgICAgaW5wdXRfbWFza19saXN0ID0gc29ydGVkKAogICAgICAgICAgICBnbG9iLmdsb2Iob3MucGF0aC5qb2luKG1hc2tfb3V0X3BhdGgsICIqLltqcEpQXVtwblBOXSpbZ0ddIikpLAogICAgICAgICAgICBrZXk9bGFtYmRhIHg6IGludChvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUoeCkpWzBdKSwKICAgICAgICApCiAgICAgICAgc2VsZi5tYXNrX2xpc3RfY3ljbGUgPSByZWFkX2ltZ3MoaW5wdXRfbWFza19saXN0KQoKICAgICAgICBzZWxmLnJlYWR5ID0gVHJ1ZQogICAgICAgIHByaW50KGYiW1dhcm1FbmdpbmVdIFJFQURZIGluIHt0aW1lLnRpbWUoKSAtIHQwOi4xZn1zIG9uIHtzZWxmLmRldmljZX0iKQogICAgICAgIHJldHVybiBUcnVlCgogICAgZGVmIF9wcm9jZXNzX2ZyYW1lcyhzZWxmLCByZXNfZnJhbWVfcXVldWUsIHZpZGVvX2xlbiwgdG1wX2RpciwgaWR4X2JveCk6CiAgICAgICAgZnJvbSBtdXNldGFsay51dGlscy5ibGVuZGluZyBpbXBvcnQgZ2V0X2ltYWdlX2JsZW5kaW5nCgogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIGlmIGlkeF9ib3hbMF0gPj0gdmlkZW9fbGVuIC0gMToKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHJlc19mcmFtZSA9IHJlc19mcmFtZV9xdWV1ZS5nZXQoYmxvY2s9VHJ1ZSwgdGltZW91dD0xKQogICAgICAgICAgICBleGNlcHQgcXVldWUuRW1wdHk6CiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgaSA9IGlkeF9ib3hbMF0KICAgICAgICAgICAgYmJveCA9IHNlbGYuY29vcmRfbGlzdF9jeWNsZVtpICUgbGVuKHNlbGYuY29vcmRfbGlzdF9jeWNsZSldCiAgICAgICAgICAgIG9yaV9mcmFtZSA9IGNvcHkuZGVlcGNvcHkoc2VsZi5mcmFtZV9saXN0X2N5Y2xlW2kgJSBsZW4oc2VsZi5mcmFtZV9saXN0X2N5Y2xlKV0pCiAgICAgICAgICAgIHgxLCB5MSwgeDIsIHkyID0gYmJveAogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICByZXNfZnJhbWUgPSBjdjIucmVzaXplKHJlc19mcmFtZS5hc3R5cGUobnAudWludDgpLCAoeDIgLSB4MSwgeTIgLSB5MSkpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBpZHhfYm94WzBdID0gaSArIDEKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIG1hc2sgPSBzZWxmLm1hc2tfbGlzdF9jeWNsZVtpICUgbGVuKHNlbGYubWFza19saXN0X2N5Y2xlKV0KICAgICAgICAgICAgbWFza19jcm9wX2JveCA9IHNlbGYubWFza19jb29yZHNfbGlzdF9jeWNsZVtpICUgbGVuKHNlbGYubWFza19jb29yZHNfbGlzdF9jeWNsZSldCiAgICAgICAgICAgIGNvbWJpbmVfZnJhbWUgPSBnZXRfaW1hZ2VfYmxlbmRpbmcob3JpX2ZyYW1lLCByZXNfZnJhbWUsIGJib3gsIG1hc2ssIG1hc2tfY3JvcF9ib3gpCiAgICAgICAgICAgICMgUGhhc2UgMS41OiBsaWdodCB1bnNoYXJwIChjaGVhcCkg4oCUIEdGUEdBTiBsYXRlciBpZiBzdGlsbCBzb2Z0CiAgICAgICAgICAgIGlmIG9zLmVudmlyb24uZ2V0KCJNVVNFVEFMS19TSEFSUEVOIiwgIjEiKSBub3QgaW4gKCIwIiwgImZhbHNlIiwgIkZhbHNlIik6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgYmx1ciA9IGN2Mi5HYXVzc2lhbkJsdXIoY29tYmluZV9mcmFtZSwgKDAsIDApLCAxLjApCiAgICAgICAgICAgICAgICAgICAgY29tYmluZV9mcmFtZSA9IGN2Mi5hZGRXZWlnaHRlZChjb21iaW5lX2ZyYW1lLCAxLjM1LCBibHVyLCAtMC4zNSwgMCkKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICAgICAgcGFzcwogICAgICAgICAgICBjdjIuaW13cml0ZShmInt0bXBfZGlyfS97c3RyKGkpLnpmaWxsKDgpfS5wbmciLCBjb21iaW5lX2ZyYW1lKQogICAgICAgICAgICBpZHhfYm94WzBdID0gaSArIDEKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgaW5mZXIoc2VsZiwgYXVkaW9fcGF0aDogc3RyLCBvdXRwdXRfbXA0OiBzdHIsIGNsaXBfa2V5OiBzdHIpIC0+IGJvb2w6CiAgICAgICAgIiIiUnVuIG9uZSBsaXAtc3luYyBqb2IgdXNpbmcgYWxyZWFkeS13YXJtIG1vZGVscy4gV3JpdGVzIHJhdyBNdXNlVGFsayBNUDQuIiIiCiAgICAgICAgaWYgbm90IHNlbGYucmVhZHk6CiAgICAgICAgICAgIHByaW50KGYiW1dhcm1FbmdpbmVdIG5vdCByZWFkeToge3NlbGYuX2xvYWRfZXJyb3J9IikKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgICAgIGZyb20gbXVzZXRhbGsudXRpbHMudXRpbHMgaW1wb3J0IGRhdGFnZW4KCiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgICAgIHRtcF9kaXIgPSBvcy5wYXRoLmpvaW4oc2VsZi5hdmF0YXJfcGF0aCwgZiJ0bXBfe2NsaXBfa2V5fSIpCiAgICAgICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKHRtcF9kaXIpOgogICAgICAgICAgICAgICAgc2h1dGlsLnJtdHJlZSh0bXBfZGlyLCBpZ25vcmVfZXJyb3JzPVRydWUpCiAgICAgICAgICAgIG9zLm1ha2VkaXJzKHRtcF9kaXIsIGV4aXN0X29rPVRydWUpCgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB3aGlzcGVyX2lucHV0X2ZlYXR1cmVzLCBsaWJyb3NhX2xlbmd0aCA9IHNlbGYuYXVkaW9fcHJvY2Vzc29yLmdldF9hdWRpb19mZWF0dXJlKAogICAgICAgICAgICAgICAgICAgIGF1ZGlvX3BhdGgsIHdlaWdodF9kdHlwZT1zZWxmLndlaWdodF9kdHlwZQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgd2hpc3Blcl9jaHVua3MgPSBzZWxmLmF1ZGlvX3Byb2Nlc3Nvci5nZXRfd2hpc3Blcl9jaHVuaygKICAgICAgICAgICAgICAgICAgICB3aGlzcGVyX2lucHV0X2ZlYXR1cmVzLAogICAgICAgICAgICAgICAgICAgIHNlbGYuZGV2aWNlLAogICAgICAgICAgICAgICAgICAgIHNlbGYud2VpZ2h0X2R0eXBlLAogICAgICAgICAgICAgICAgICAgIHNlbGYud2hpc3BlciwKICAgICAgICAgICAgICAgICAgICBsaWJyb3NhX2xlbmd0aCwKICAgICAgICAgICAgICAgICAgICBmcHM9c2VsZi5mcHMsCiAgICAgICAgICAgICAgICAgICAgYXVkaW9fcGFkZGluZ19sZW5ndGhfbGVmdD0yLAogICAgICAgICAgICAgICAgICAgIGF1ZGlvX3BhZGRpbmdfbGVuZ3RoX3JpZ2h0PTIsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICB2aWRlb19udW0gPSBsZW4od2hpc3Blcl9jaHVua3MpCiAgICAgICAgICAgICAgICBpZiB2aWRlb19udW0gPCAxOgogICAgICAgICAgICAgICAgICAgIHByaW50KCJbV2FybUVuZ2luZV0gbm8gd2hpc3BlciBjaHVua3MgZnJvbSBhdWRpbyIpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgICAgICAgICAgICAgcmVzX2ZyYW1lX3F1ZXVlOiBxdWV1ZS5RdWV1ZSA9IHF1ZXVlLlF1ZXVlKCkKICAgICAgICAgICAgICAgIGlkeF9ib3ggPSBbMF0KICAgICAgICAgICAgICAgIHByb2Nlc3NfdGhyZWFkID0gdGhyZWFkaW5nLlRocmVhZCgKICAgICAgICAgICAgICAgICAgICB0YXJnZXQ9c2VsZi5fcHJvY2Vzc19mcmFtZXMsCiAgICAgICAgICAgICAgICAgICAgYXJncz0ocmVzX2ZyYW1lX3F1ZXVlLCB2aWRlb19udW0sIHRtcF9kaXIsIGlkeF9ib3gpLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgcHJvY2Vzc190aHJlYWQuc3RhcnQoKQoKICAgICAgICAgICAgICAgIGdlbiA9IGRhdGFnZW4od2hpc3Blcl9jaHVua3MsIHNlbGYuaW5wdXRfbGF0ZW50X2xpc3RfY3ljbGUsIHNlbGYuYmF0Y2hfc2l6ZSkKICAgICAgICAgICAgICAgIGZvciB3aGlzcGVyX2JhdGNoLCBsYXRlbnRfYmF0Y2ggaW4gdHFkbSgKICAgICAgICAgICAgICAgICAgICBnZW4sIHRvdGFsPWludChucC5jZWlsKGZsb2F0KHZpZGVvX251bSkgLyBzZWxmLmJhdGNoX3NpemUpKSwgZGVzYz0iV2FybUluZmVyIgogICAgICAgICAgICAgICAgKToKICAgICAgICAgICAgICAgICAgICBhdWRpb19mZWF0dXJlX2JhdGNoID0gc2VsZi5wZSh3aGlzcGVyX2JhdGNoLnRvKHNlbGYuZGV2aWNlKSkKICAgICAgICAgICAgICAgICAgICBsYXRlbnRfYmF0Y2ggPSBsYXRlbnRfYmF0Y2gudG8oZGV2aWNlPXNlbGYuZGV2aWNlLCBkdHlwZT1zZWxmLnVuZXQubW9kZWwuZHR5cGUpCiAgICAgICAgICAgICAgICAgICAgcHJlZF9sYXRlbnRzID0gc2VsZi51bmV0Lm1vZGVsKAogICAgICAgICAgICAgICAgICAgICAgICBsYXRlbnRfYmF0Y2gsIHNlbGYudGltZXN0ZXBzLCBlbmNvZGVyX2hpZGRlbl9zdGF0ZXM9YXVkaW9fZmVhdHVyZV9iYXRjaAogICAgICAgICAgICAgICAgICAgICkuc2FtcGxlCiAgICAgICAgICAgICAgICAgICAgcHJlZF9sYXRlbnRzID0gcHJlZF9sYXRlbnRzLnRvKGRldmljZT1zZWxmLmRldmljZSwgZHR5cGU9c2VsZi52YWUudmFlLmR0eXBlKQogICAgICAgICAgICAgICAgICAgIHJlY29uID0gc2VsZi52YWUuZGVjb2RlX2xhdGVudHMocHJlZF9sYXRlbnRzKQogICAgICAgICAgICAgICAgICAgIGZvciByZXNfZnJhbWUgaW4gcmVjb246CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc19mcmFtZV9xdWV1ZS5wdXQocmVzX2ZyYW1lKQoKICAgICAgICAgICAgICAgIHByb2Nlc3NfdGhyZWFkLmpvaW4oKQoKICAgICAgICAgICAgICAgICMgRW5jb2RlIGZyYW1lcyDihpIgc2lsZW50IHZpZGVvLCB0aGVuIG11eCB3aXRoIGlucHV0IGF1ZGlvIChzZXJ2ZXIgcmVtdXhlcyBHZW1pbmkgYWdhaW4pCiAgICAgICAgICAgICAgICB0ZW1wX3ZpZCA9IG9zLnBhdGguam9pbihzZWxmLmF2YXRhcl9wYXRoLCBmInRlbXBfe2NsaXBfa2V5fS5tcDQiKQogICAgICAgICAgICAgICAgY21kX2ltZzJ2aWRlbyA9IFsKICAgICAgICAgICAgICAgICAgICAiZmZtcGVnIiwKICAgICAgICAgICAgICAgICAgICAiLXkiLAogICAgICAgICAgICAgICAgICAgICItdiIsCiAgICAgICAgICAgICAgICAgICAgIndhcm5pbmciLAogICAgICAgICAgICAgICAgICAgICItciIsCiAgICAgICAgICAgICAgICAgICAgc3RyKHNlbGYuZnBzKSwKICAgICAgICAgICAgICAgICAgICAiLWYiLAogICAgICAgICAgICAgICAgICAgICJpbWFnZTIiLAogICAgICAgICAgICAgICAgICAgICItaSIsCiAgICAgICAgICAgICAgICAgICAgZiJ7dG1wX2Rpcn0vJTA4ZC5wbmciLAogICAgICAgICAgICAgICAgICAgICItdmNvZGVjIiwKICAgICAgICAgICAgICAgICAgICAibGlieDI2NCIsCiAgICAgICAgICAgICAgICAgICAgIi12ZiIsCiAgICAgICAgICAgICAgICAgICAgImZvcm1hdD15dXY0MjBwIiwKICAgICAgICAgICAgICAgICAgICAiLWNyZiIsCiAgICAgICAgICAgICAgICAgICAgIjE4IiwKICAgICAgICAgICAgICAgICAgICB0ZW1wX3ZpZCwKICAgICAgICAgICAgICAgIF0KICAgICAgICAgICAgICAgIHIxID0gc3VicHJvY2Vzcy5ydW4oY21kX2ltZzJ2aWRlbywgc3Rkb3V0PXN1YnByb2Nlc3MuUElQRSwgc3RkZXJyPXN1YnByb2Nlc3MuUElQRSkKICAgICAgICAgICAgICAgIGlmIHIxLnJldHVybmNvZGUgIT0gMCBvciBub3Qgb3MucGF0aC5leGlzdHModGVtcF92aWQpOgogICAgICAgICAgICAgICAgICAgIHByaW50KGYiW1dhcm1FbmdpbmVdIGltZzJ2aWRlbyBmYWlsZWQ6IHsocjEuc3RkZXJyIG9yIGInJylbLTQwMDpdfSIpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgICAgICAgICAgICAgb3MubWFrZWRpcnMob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChvdXRwdXRfbXA0KSkgb3IgIi4iLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICAgICAgY21kX211eCA9IFsKICAgICAgICAgICAgICAgICAgICAiZmZtcGVnIiwKICAgICAgICAgICAgICAgICAgICAiLXkiLAogICAgICAgICAgICAgICAgICAgICItdiIsCiAgICAgICAgICAgICAgICAgICAgIndhcm5pbmciLAogICAgICAgICAgICAgICAgICAgICItaSIsCiAgICAgICAgICAgICAgICAgICAgYXVkaW9fcGF0aCwKICAgICAgICAgICAgICAgICAgICAiLWkiLAogICAgICAgICAgICAgICAgICAgIHRlbXBfdmlkLAogICAgICAgICAgICAgICAgICAgICItYzp2IiwKICAgICAgICAgICAgICAgICAgICAiY29weSIsCiAgICAgICAgICAgICAgICAgICAgIi1jOmEiLAogICAgICAgICAgICAgICAgICAgICJhYWMiLAogICAgICAgICAgICAgICAgICAgICItc2hvcnRlc3QiLAogICAgICAgICAgICAgICAgICAgIG91dHB1dF9tcDQsCiAgICAgICAgICAgICAgICBdCiAgICAgICAgICAgICAgICByMiA9IHN1YnByb2Nlc3MucnVuKGNtZF9tdXgsIHN0ZG91dD1zdWJwcm9jZXNzLlBJUEUsIHN0ZGVycj1zdWJwcm9jZXNzLlBJUEUpCiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgb3MucmVtb3ZlKHRlbXBfdmlkKQogICAgICAgICAgICAgICAgZXhjZXB0IE9TRXJyb3I6CiAgICAgICAgICAgICAgICAgICAgcGFzcwogICAgICAgICAgICAgICAgaWYgcjIucmV0dXJuY29kZSAhPSAwIG9yIG5vdCBvcy5wYXRoLmV4aXN0cyhvdXRwdXRfbXA0KToKICAgICAgICAgICAgICAgICAgICBwcmludChmIltXYXJtRW5naW5lXSBtdXggZmFpbGVkOiB7KHIyLnN0ZGVyciBvciBiJycpWy00MDA6XX0iKQogICAgICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQoKICAgICAgICAgICAgICAgICMgQWxzbyBjb3B5IGludG8gTXVzZVRhbGsgdmlkX291dHB1dCBmb3IgZGVidWdnaW5nCiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgc2h1dGlsLmNvcHlmaWxlKAogICAgICAgICAgICAgICAgICAgICAgICBvdXRwdXRfbXA0LCBvcy5wYXRoLmpvaW4oc2VsZi52aWRlb19vdXRfcGF0aCwgZiJ7Y2xpcF9rZXl9Lm1wNCIpCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICBwYXNzCgogICAgICAgICAgICAgICAgcHJpbnQoCiAgICAgICAgICAgICAgICAgICAgZiJbV2FybUVuZ2luZV0gT0sgY2xpcD17Y2xpcF9rZXl9IGZyYW1lcz17dmlkZW9fbnVtfSAiCiAgICAgICAgICAgICAgICAgICAgZiJpbiB7dGltZS50aW1lKCkgLSB0MDouMmZ9cyDihpIge291dHB1dF9tcDR9IgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICAgICAgcHJpbnQoZiJbV2FybUVuZ2luZV0gaW5mZXIgZXJyb3I6IHtlfSIpCiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICAgICAgZmluYWxseToKICAgICAgICAgICAgICAgIHNodXRpbC5ybXRyZWUodG1wX2RpciwgaWdub3JlX2Vycm9ycz1UcnVlKQoKCl9FTkdJTkU6IE9wdGlvbmFsW1dhcm1NdXNlVGFsa0VuZ2luZV0gPSBOb25lCl9FTkdJTkVfTE9DSyA9IHRocmVhZGluZy5Mb2NrKCkKCgpkZWYgZ2V0X3dhcm1fZW5naW5lKAogICAgdmlkZW9fcGF0aDogc3RyLAogICAgYmJveF9zaGlmdDogaW50ID0gMCwKICAgIGJhdGNoX3NpemU6IGludCA9IDE2LAopIC0+IE9wdGlvbmFsW1dhcm1NdXNlVGFsa0VuZ2luZV06CiAgICBnbG9iYWwgX0VOR0lORQogICAgd2l0aCBfRU5HSU5FX0xPQ0s6CiAgICAgICAgaWYgX0VOR0lORSBpcyBub3QgTm9uZSBhbmQgX0VOR0lORS5yZWFkeToKICAgICAgICAgICAgcmV0dXJuIF9FTkdJTkUKICAgICAgICBlbmcgPSBXYXJtTXVzZVRhbGtFbmdpbmUoCiAgICAgICAgICAgIHZpZGVvX3BhdGg9dmlkZW9fcGF0aCwKICAgICAgICAgICAgYmJveF9zaGlmdD1iYm94X3NoaWZ0LAogICAgICAgICAgICBiYXRjaF9zaXplPWJhdGNoX3NpemUsCiAgICAgICAgICAgIHZlcnNpb249InYxNSIsCiAgICAgICAgKQogICAgICAgIGlmIGVuZy5sb2FkKCk6CiAgICAgICAgICAgIF9FTkdJTkUgPSBlbmcKICAgICAgICAgICAgcmV0dXJuIF9FTkdJTkUKICAgICAgICByZXR1cm4gTm9uZQo="
SERVER_B64 = "..."
with open('/content/MuseTalk/musetalk_warm_engine.py', 'wb') as f:
    f.write(base64.b64decode(WARM_B64))
with open('/content/MuseTalk/musetalk_server.py', 'wb') as f:
    f.write(base64.b64decode(SERVER_B64))
print('✅ musetalk_warm_engine.py + musetalk_server.py v3.0 written')


## 5. Launch FastAPI Server & Public Cloudflare Tunnel

In [ ]:
# 5. Launch FastAPI Server & Public Cloudflare Tunnel
import os, sys, time, subprocess, urllib.request, re
%cd /content/MuseTalk

!pkill -9 -f uvicorn || true
!pkill -9 -f cloudflared || true

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

ENV_DIR = "/root/.local/share/mamba/envs/py310" if os.path.exists("/root/.local/share/mamba/envs/py310") else "/usr/local/micromamba/envs/py310"
ENV_PY = f"{ENV_DIR}/bin/python"

print("🚀 Starting MuseTalk FastAPI Server on port 8000...")
subprocess.Popen(f"{ENV_PY} -m uvicorn musetalk_server:app --host 0.0.0.0 --port 8000 > /content/server.log 2>&1", shell=True)

# Verify local server responded
healthy = False
for i in range(12):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as res:
            data = res.read().decode()
            print("✅ MuseTalk FastAPI Server is UP & HEALTHY:", data)
            try:
                req = urllib.request.Request("http://127.0.0.1:8000/api/warmup", method="POST")
                with urllib.request.urlopen(req, timeout=180) as wr:
                    print("🔥 Warmup:", wr.read().decode())
            except Exception as we:
                print("⚠️ Warmup deferred:", we)
            healthy = True
            break
    except Exception:
        pass

if not healthy:
    print("⚠️ MuseTalk server took longer to start. Recent logs:")
    !tail -n 20 /content/server.log

print("\n🌐 Establishing Cloudflare Secure Public Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*72)
    print("🎉 MUSE TALK SERVER IS LIVE & READY FOR SABCQ! 🎉")
    print("="*72)
    print("👉 Step 1: Copy this URL:")
    print(f"\n   {tunnel_url}\n")
    print("👉 Step 2: Open aiqtest_backend/.env in your project and update:")
    print(f"\n   AVATAR_SERVICE_URL={tunnel_url}\n")
    print("="*72)
    print("🟢 STATUS: Waiting for counseling voice audio from SABCQ AI Counselor...")
    print("⚠️ NOTE: The spinner at the top left of this cell is NORMAL.")
    print("         The tunnel keeps running to serve your requests. Keep this tab open!")
    print("="*72 + "\n")
else:
    print("⚠️ Could not parse tunnel URL automatically. Tunnel logs:")

# Keep the tunnel alive and running
while True:
    line = tunnel_proc.stdout.readline()
    if not line and tunnel_proc.poll() is not None:
        print("⚠️ Tunnel process exited!")
        break
    time.sleep(1)
